# 04 — Moving exclusive anchors on cluttered images

Experiment 03 showed that anchors which are the same for every image collapse, content-driven
anchors converge with depth, and exclusive regions stay diverse. This notebook tests the
proposed fix: **anchors that move to important regions but keep exclusive regions**.

**Anchor pathway (`VoronoiRouter`).** Each anchor has a 2-D position. Every patch belongs
to its nearest anchor (a Voronoi partition), so regions stay exclusive. With `move=True`,
a learned importance map pulls each anchor to the importance-weighted centroid of its
region at every block (one weighted Lloyd / k-means step). A centroid always lies inside
its own region, so anchors cannot merge. The forward pass uses the hard assignment; the
backward pass uses a soft distance-softmax (straight-through), so positions and
importance are trained by the task loss. With `move=False` the anchors stay on a fixed
grid, which reproduces Partition RART with the same code.

**Task (Cluttered CIFAR).** A 64×64 CIFAR object is pasted at a random position on a
256×256 canvas, over 8 distractor fragments cut from other images. The object covers
6.25% of the canvas, so fixed anchors spend most of their budget on clutter. The object's
box is known, so we can measure how much anchor attention lands on it.

**Models.** `free` (RART-style learned queries, softmax over all patches — collapsed in
experiment 03), `fixed` (Voronoi, frozen grid), `moving` (Voronoi, importance-weighted Lloyd
step), and `focus` (a quarter of the anchors on a coarse fixed grid for coverage, the
rest placed at importance peaks anywhere in the image), each with 4 and 16 anchors.

`moving` turned out to be a coverage rule: its importance map finds the object, but the
anchors move only ~1 patch because each Lloyd step stays inside the anchor's own
region. `focus` separates the two jobs — coverage anchors keep every region summarised,
focus anchors jump to high-importance content anywhere in the image — while all anchors
still share one exclusive partition. Focus anchors can leave a region empty, so the
pooling step clamps its normaliser at 1 (see `VoronoiRouter.forward`).

Environment overrides: `RART_MODELS`, `RART_ANCHOR_COUNTS` (square numbers),
`RART_EPOCHS`, `RART_SEED`, `RART_DATASET` (`cifar10` or `cifar100`),
`RART_GRAD_CLIP` (gradient-norm clipping threshold; default 0 = off, as in the reported runs),
`RART_SMOKE=1`. Runs with different clipping settings are saved to different results
files. Per-epoch gradient-norm statistics are logged for every run.

In [ ]:
import gc
import json
import math
import os
import random
import time
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision


def env(name, default):
    return os.environ.get(name, default)


SEED = int(env("RART_SEED", "42"))
SMOKE = env("RART_SMOKE", "0") == "1"
NUM_EPOCHS = 1 if SMOKE else int(env("RART_EPOCHS", "10"))
MODELS = [m.strip() for m in env("RART_MODELS", "free,fixed,moving,focus").split(",") if m.strip()]
ANCHOR_COUNTS = [int(a) for a in env("RART_ANCHOR_COUNTS", "4,16").split(",")]
DATASET = env("RART_DATASET", "cifar10")
GRAD_CLIP = float(env("RART_GRAD_CLIP", "0"))
KEEP_RATIO = float(env("RART_KEEP_RATIO", "1.0"))
SKIP_FROM = int(env("RART_SKIP_FROM", "2"))
SKIP_SCORE = env("RART_SKIP_SCORE", "importance")

CANVAS, OBJECT_SIZE, NUM_DISTRACTORS, CROP, DISTRACTOR_SIZE = 256, 64, 8, 16, 32
PATCH_SIZE, BATCH_SIZE = 16, 128
GRID = CANVAS // PATCH_SIZE
EMBED_DIM, DEPTH, HEADS, MLP_RATIO, WINDOW = 384, 6, 6, 3.0, 8
ANCHOR_DIM, ANCHOR_HEADS = 128, 4
METRIC_IMAGES = 256 if SMOKE else 1024
TEST_LAYOUT_SEED = 1234

WORK_DIR = env(
    "RART_WORK_DIR",
    "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("working"),
)
os.makedirs(WORK_DIR, exist_ok=True)
SKIP_TAG = f"_keep{KEEP_RATIO:g}-{SKIP_SCORE}-from{SKIP_FROM}" if KEEP_RATIO < 1 else ""
RUN_NAME = f"step1_{DATASET}_seed{SEED}{f'_clip{GRAD_CLIP:g}' if GRAD_CLIP > 0 else ''}{SKIP_TAG}{'_smoke' if SMOKE else ''}"
RESULTS_PATH = os.path.join(WORK_DIR, RUN_NAME + ".json")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

for count in ANCHOR_COUNTS:
    if math.isqrt(count) ** 2 != count:
        raise ValueError(f"anchor counts must be square numbers, got {count}")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print("PyTorch:", torch.__version__, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("Models:", MODELS, "| anchors:", ANCHOR_COUNTS, "| epochs:", NUM_EPOCHS, "| dataset:", DATASET,
      "| grad clip:", GRAD_CLIP,
      "| seed:", SEED, "(SMOKE TEST)" if SMOKE else "")
print("Results file:", RESULTS_PATH)

## Cluttered CIFAR

Images are composed on the GPU per batch. Training layouts are random every epoch; the
test layouts are fixed by `TEST_LAYOUT_SEED`, so every model sees the same test canvases.
Distractors come from other images of the same split.

In [ ]:
DATASETS = {
    "cifar10": (torchvision.datasets.CIFAR10, "cifar-10-batches-py", 10,
                (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    "cifar100": (torchvision.datasets.CIFAR100, "cifar-100-python", 100,
                 (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
}
dataset_cls, dataset_folder, NUM_CLASSES, mean, std = DATASETS[DATASET]


def find_data_root():
    if os.path.isdir("/kaggle/input"):
        for root, dirs, _ in os.walk("/kaggle/input"):
            if dataset_folder in dirs:
                return root, False
    root = os.path.join(WORK_DIR, "data")
    return root, not os.path.isdir(os.path.join(root, dataset_folder))


data_root, need_download = find_data_root()


def load_split(train):
    ds = dataset_cls(data_root, train=train, download=need_download)
    images = torch.from_numpy(ds.data).permute(0, 3, 1, 2).contiguous()
    labels = torch.tensor(ds.targets, dtype=torch.long)
    return images, labels


train_x, train_y = load_split(True)
test_x, test_y = load_split(False)
if SMOKE:
    train_x, train_y = train_x[:512], train_y[:512]
    test_x, test_y = test_x[:256], test_y[:256]
train_x, train_y, test_x, test_y = (t.to(DEVICE) for t in (train_x, train_y, test_x, test_y))

MEAN = torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor(std, device=DEVICE).view(1, 3, 1, 1)
CHANNELS = torch.arange(3, device=DEVICE)


def paste(canvas, tiles, batch_index, top, left):
    height, width = tiles.shape[-2:]
    rows = top[:, None] + torch.arange(height, device=DEVICE)
    cols = left[:, None] + torch.arange(width, device=DEVICE)
    canvas[batch_index[:, None, None, None], CHANNELS[None, :, None, None],
           rows[:, None, :, None], cols[:, None, None, :]] = tiles


def compose(objects, pool, generator, flip):
    """objects: uint8 (B, 3, 32, 32). Returns normalised canvases and object boxes
    (B, 4) as (top, left, bottom, right) in pixels."""
    batch = objects.shape[0]
    canvas = torch.zeros(batch, 3, CANVAS, CANVAS, device=DEVICE)

    count = batch * NUM_DISTRACTORS
    source = pool[torch.randint(len(pool), (count,), generator=generator).to(DEVICE)].float().div_(255)
    offsets = torch.randint(32 - CROP + 1, (count, 2), generator=generator).to(DEVICE)
    rows = offsets[:, :1] + torch.arange(CROP, device=DEVICE)
    cols = offsets[:, 1:] + torch.arange(CROP, device=DEVICE)
    crops = source[torch.arange(count, device=DEVICE)[:, None, None, None], CHANNELS[None, :, None, None],
                   rows[:, None, :, None], cols[:, None, None, :]]
    crops = F.interpolate(crops, size=(DISTRACTOR_SIZE, DISTRACTOR_SIZE), mode="bilinear", align_corners=False)
    places = torch.randint(CANVAS - DISTRACTOR_SIZE + 1, (count, 2), generator=generator).to(DEVICE)
    paste(canvas, crops, torch.arange(batch, device=DEVICE).repeat_interleave(NUM_DISTRACTORS),
          places[:, 0], places[:, 1])

    objects = objects.float().div_(255)
    if flip:
        mirror = (torch.rand(batch, generator=generator) < 0.5).to(DEVICE)
        objects = torch.where(mirror.view(-1, 1, 1, 1), objects.flip(-1), objects)
    objects = F.interpolate(objects, size=(OBJECT_SIZE, OBJECT_SIZE), mode="bilinear", align_corners=False)
    corners = torch.randint(CANVAS - OBJECT_SIZE + 1, (batch, 2), generator=generator).to(DEVICE)
    paste(canvas, objects, torch.arange(batch, device=DEVICE), corners[:, 0], corners[:, 1])

    boxes = torch.cat([corners, corners + OBJECT_SIZE], dim=1)
    return (canvas - MEAN) / STD, boxes


def iterate_batches(images, labels, train, generator=None):
    n = images.shape[0]
    if train:
        order = torch.randperm(n, generator=generator).to(DEVICE)
    else:
        order = torch.arange(n, device=DEVICE)
        generator = torch.Generator().manual_seed(TEST_LAYOUT_SEED)
    for start in range(0, n, BATCH_SIZE):
        idx = order[start:start + BATCH_SIZE]
        x, boxes = compose(images[idx], images, generator, flip=train)
        yield x, labels[idx], boxes


def object_patch_mask(boxes):
    """(B, N) bool: patches whose centre lies inside the object box."""
    centres = (torch.arange(GRID, device=DEVICE) + 0.5) * PATCH_SIZE
    inside_rows = (centres[None] >= boxes[:, :1]) & (centres[None] < boxes[:, 2:3])
    inside_cols = (centres[None] >= boxes[:, 1:2]) & (centres[None] < boxes[:, 3:4])
    return (inside_rows[:, :, None] & inside_cols[:, None, :]).flatten(1)


sample_x, sample_y, sample_boxes = next(iterate_batches(test_x, test_y, train=False))
print("Canvas batch:", tuple(sample_x.shape), "| object patches per image:",
      object_patch_mask(sample_boxes).sum(1).float().mean().item(), "of", GRID * GRID)

## Shared backbone

In [ ]:
def split_heads(t, heads):
    batch, length, dim = t.shape
    return t.view(batch, length, heads, dim // heads).transpose(1, 2)


def merge_heads(t):
    batch, heads, length, dim = t.shape
    return t.transpose(1, 2).reshape(batch, length, heads * dim)


class SelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        q, k, v = (split_heads(t, self.heads) for t in self.qkv(x).chunk(3, dim=-1))
        return self.proj(merge_heads(F.scaled_dot_product_attention(q, k, v)))


class LocalWindowAttention(nn.Module):
    def __init__(self, dim, heads, window):
        super().__init__()
        self.window = window
        self.attention = SelfAttention(dim, heads)

    def forward(self, x, grid):
        batch, num_patches, dim = x.shape
        w, n = self.window, grid // self.window
        windows = x.view(batch, n, w, n, w, dim).permute(0, 1, 3, 2, 4, 5).reshape(-1, w * w, dim)
        windows = self.attention(windows)
        return windows.view(batch, n, n, w, w, dim).permute(0, 1, 3, 2, 4, 5).reshape(batch, num_patches, dim)


def grid_positions(side):
    """Cell centres of a side × side grid in normalised [0, 1] coordinates, row-major."""
    centres = (torch.arange(side, dtype=torch.float32) + 0.5) / side
    rows, cols = torch.meshgrid(centres, centres, indexing="ij")
    return torch.stack([rows, cols], dim=-1).reshape(-1, 2)

## Anchor pathways

Routers take the normalised patches and the state `(anchors, positions)` and return a
patch update and the new state. With `record` set, they keep each anchor's pooling
distribution over patches, its pooled content, and its position.

In [ ]:
class AnchorRouter(nn.Module):
    def __init__(self, num_anchors):
        super().__init__()
        da = ANCHOR_DIM
        self.num_anchors = num_anchors
        self.heads = ANCHOR_HEADS
        self.scale = (da // ANCHOR_HEADS) ** -0.5
        self.gather_in = nn.Linear(EMBED_DIM, da)
        self.gather_q = nn.Linear(da, da)
        self.gather_kv = nn.Linear(da, da * 2)
        self.gather_out = nn.Linear(da, da)
        self.anchor_norm = nn.LayerNorm(da)
        self.anchor_attention = SelfAttention(da, ANCHOR_HEADS)
        self.scatter_norm = nn.LayerNorm(da)
        self.scatter_in = nn.Linear(EMBED_DIM, da)
        self.scatter_q = nn.Linear(da, da)
        self.scatter_kv = nn.Linear(da, da * 2)
        self.scatter_out = nn.Linear(da, EMBED_DIM)
        self.record = False
        self.last = None

    def keep(self, pooling, pooled, positions):
        if self.record:
            self.last = (pooling.detach().float(), pooled.detach().float(),
                         None if positions is None else positions.detach().float())

    def gather_logits(self, x, anchors):
        k, v = (split_heads(t, self.heads) for t in self.gather_kv(self.gather_in(x)).chunk(2, dim=-1))
        q = split_heads(self.gather_q(anchors), self.heads)
        return (q @ k.transpose(-2, -1)).float() * self.scale, v

    def update_anchors(self, anchors, weights, v):
        pooled = merge_heads(weights.to(v.dtype) @ v)
        anchors = anchors + self.gather_out(pooled)
        anchors = anchors + self.anchor_attention(self.anchor_norm(anchors))
        return anchors, pooled

    def scatter_inputs(self, x, anchors):
        k, v = (split_heads(t, self.heads) for t in self.scatter_kv(self.scatter_norm(anchors)).chunk(2, dim=-1))
        q = split_heads(self.scatter_q(self.scatter_in(x)), self.heads)
        return q, k, v


class FreeRouter(AnchorRouter):
    """RART-16/32 routing: learned anchor queries, softmax over all patches."""

    def forward(self, x, state):
        anchors, positions = state
        logits, v = self.gather_logits(x, anchors)
        weights = logits.softmax(dim=-1)
        anchors, pooled = self.update_anchors(anchors, weights, v)
        q, k, v = self.scatter_inputs(x, anchors)
        update = self.scatter_out(merge_heads(F.scaled_dot_product_attention(q, k, v)))
        self.keep(weights.mean(1), pooled, None)
        return update, (anchors, positions)


class VoronoiRouter(AnchorRouter):
    """Anchors with 2-D positions and an exclusive Voronoi partition of the patches.
    move=False keeps the positions on a fixed grid (Partition RART); move=True pulls each
    anchor to the importance-weighted centroid of its region every block."""

    def __init__(self, num_anchors, move):
        super().__init__(num_anchors)
        self.move = move
        self.register_buffer("coords", grid_positions(GRID), persistent=False)
        self.tau = (0.5 / math.isqrt(num_anchors)) ** 2
        self.position_embed = nn.Sequential(
            nn.Linear(2, ANCHOR_DIM), nn.GELU(), nn.Linear(ANCHOR_DIM, ANCHOR_DIM)
        )
        if move:
            self.importance = nn.Linear(EMBED_DIM, 1)

    def assignment(self, positions):
        """(B, A, N) one-hot patch → anchor assignment in the forward pass, with the
        gradient of a soft distance-softmax (straight-through)."""
        distance = (positions[:, :, None, :] - self.coords[None, None]).square().sum(dim=-1)
        soft = (-distance / self.tau).softmax(dim=1)
        hard = F.one_hot(soft.argmax(dim=1), self.num_anchors).transpose(1, 2).to(soft.dtype)
        return hard + soft - soft.detach()

    def place(self, x, positions):
        """Returns the anchor positions for this block and the patch importance
        (None when the router does not use importance)."""
        if not self.move:
            return positions, None
        importance = F.softplus(self.importance(x).squeeze(-1).float()) + 1e-3
        mass = self.assignment(positions) * importance[:, None, :]
        total = mass.sum(dim=-1, keepdim=True)
        centroid = (mass @ self.coords) / total.clamp_min(1e-6)
        return torch.where(total > 1e-4, centroid, positions), importance

    def forward(self, x, state):
        anchors, positions = state
        positions, importance = self.place(x, positions)
        self.current_importance = importance

        assign = self.assignment(positions)
        hard = assign.detach() > 0.5
        located = anchors + self.position_embed(positions).to(anchors.dtype)
        logits, v = self.gather_logits(x, located)
        if importance is not None:
            logits = logits + importance.log()[:, None, None, :]
        peak = logits.masked_fill(~hard[:, None], float("-inf")).amax(dim=-1, keepdim=True)
        peak = torch.nan_to_num(peak.detach(), neginf=0.0)
        scores = (logits - peak).clamp_max(0).exp() * assign[:, None]
        # A non-empty region always sums to >= 1 (its top patch scores exp(0) = 1), so the
        # clamp only affects empty regions, where a tiny denominator would blow up the
        # straight-through gradient (focus anchors can leave a region empty).
        weights = scores / scores.sum(dim=-1, keepdim=True).clamp_min(1.0)
        anchors, pooled = self.update_anchors(anchors, weights, v)

        q, k, v = self.scatter_inputs(x, anchors)
        gate = torch.sigmoid((q @ k.transpose(-2, -1)).float() * self.scale)
        read = (gate * assign.transpose(1, 2)[:, None]).to(v.dtype)
        update = self.scatter_out(merge_heads(read @ v))

        self.keep(weights.mean(1), pooled, positions)
        return update, (anchors, positions)


class FocusRouter(VoronoiRouter):
    """Coverage + focus anchors. A quarter of the anchors (at least one) stay on a coarse
    fixed grid so every region keeps a global summary; the rest are placed each block at
    the peaks of the learned importance map, anywhere in the image. Peaks are found by a
    soft-argmax, and each chosen peak is suppressed with a Gaussian so the next focus
    anchor lands elsewhere. Each focus anchor also has a spread-out home position with a
    weak Gaussian prior, so with a flat importance map the anchors stay spread instead of
    all starting at the image centre; a strong importance peak anywhere overrides it.
    All anchors then share one exclusive Voronoi partition."""

    SUPPRESSION_STRENGTH = 10.0
    SUPPRESSION_WIDTH = 1.5 / GRID
    HOME_WIDTH = 0.35

    def __init__(self, num_anchors):
        super().__init__(num_anchors, move=True)
        coverage_side = max(1, math.isqrt(num_anchors // 4))
        self.num_focus = num_anchors - coverage_side ** 2
        self.register_buffer("coverage", grid_positions(coverage_side), persistent=False)
        spread = torch.linspace(0, num_anchors - 1, self.num_focus).round().long()
        homes = grid_positions(math.isqrt(num_anchors))[spread]
        home_distance = (self.coords[None] - homes[:, None]).square().sum(dim=-1)
        self.register_buffer("home_prior", -home_distance / (2 * self.HOME_WIDTH ** 2), persistent=False)

    def place(self, x, positions):
        scores = self.importance(x).squeeze(-1).float()
        importance = F.softplus(scores) + 1e-3
        peaks = []
        for k in range(self.num_focus):
            peak = (scores + self.home_prior[k]).softmax(dim=-1) @ self.coords
            peaks.append(peak)
            distance = (self.coords[None] - peak[:, None]).square().sum(dim=-1)
            scores = scores - self.SUPPRESSION_STRENGTH * torch.exp(-distance / (2 * self.SUPPRESSION_WIDTH ** 2))
        coverage = self.coverage.expand(x.shape[0], -1, -1)
        return torch.cat([coverage, torch.stack(peaks, dim=1)], dim=1), importance


ROUTERS = {
    "free": lambda a: FreeRouter(a),
    "fixed": lambda a: VoronoiRouter(a, move=False),
    "moving": lambda a: VoronoiRouter(a, move=True),
    "focus": lambda a: FocusRouter(a),
}


class Block(nn.Module):
    """Local window attention → anchor pathway → FFN. From block SKIP_FROM on, when
    KEEP_RATIO < 1, only the top KEEP_RATIO of patches (by SKIP_SCORE) go through the FFN;
    the rest skip it but have already received their anchor's global update."""

    def __init__(self, router, index):
        super().__init__()
        self.index = index
        self.local_norm = nn.LayerNorm(EMBED_DIM)
        self.local = LocalWindowAttention(EMBED_DIM, HEADS, WINDOW)
        self.global_norm = nn.LayerNorm(EMBED_DIM)
        self.router = router
        self.ffn_norm = nn.LayerNorm(EMBED_DIM)
        hidden = int(EMBED_DIM * MLP_RATIO)
        self.ffn = nn.Sequential(nn.Linear(EMBED_DIM, hidden), nn.GELU(), nn.Linear(hidden, EMBED_DIM))

    def forward(self, x, state):
        x = x + self.local(self.local_norm(x), GRID)
        update, state = self.router(self.global_norm(x), state)
        x = x + update
        if KEEP_RATIO < 1 and self.index >= SKIP_FROM:
            x = x + self.sparse_ffn(x)
        else:
            x = x + self.ffn(self.ffn_norm(x))
        return x, state

    def sparse_ffn(self, x):
        batch, num_patches, dim = x.shape
        keep = max(1, math.ceil(num_patches * KEEP_RATIO))
        importance = getattr(self.router, "current_importance", None)
        if SKIP_SCORE == "importance":
            if importance is None:
                raise ValueError("SKIP_SCORE=importance needs a router with an importance map")
            score = importance
        elif SKIP_SCORE == "norm":
            score = x.detach().float().norm(dim=-1)
        elif SKIP_SCORE == "random":
            score = torch.rand(batch, num_patches, device=x.device)
        else:
            raise ValueError(f"unknown SKIP_SCORE {SKIP_SCORE}")
        index = score.topk(keep, dim=1).indices
        kept = x.gather(1, index[:, :, None].expand(-1, -1, dim))
        out = self.ffn(self.ffn_norm(kept))
        if SKIP_SCORE == "importance":
            chosen = importance.gather(1, index)
            out = out * (chosen / chosen.detach())[:, :, None].to(out.dtype)
        return torch.zeros(batch, num_patches, dim, dtype=out.dtype, device=x.device).scatter(
            1, index[:, :, None].expand(-1, -1, dim), out
        )


class AnchorViT(nn.Module):
    def __init__(self, kind, num_anchors):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, EMBED_DIM, PATCH_SIZE, PATCH_SIZE)
        self.pos_embed = nn.Parameter(torch.randn(1, GRID * GRID, EMBED_DIM) * 0.02)
        self.anchor_init = nn.Parameter(torch.randn(1, num_anchors, ANCHOR_DIM) * 0.02)
        self.register_buffer("start_positions", grid_positions(math.isqrt(num_anchors)), persistent=False)
        self.blocks = nn.ModuleList(Block(ROUTERS[kind](num_anchors), i) for i in range(DEPTH))
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, NUM_CLASSES)

    def routers(self):
        return [block.router for block in self.blocks]

    def forward(self, images):
        x = self.patch_embed(images).flatten(2).transpose(1, 2) + self.pos_embed
        batch = x.shape[0]
        state = (self.anchor_init.expand(batch, -1, -1),
                 self.start_positions.expand(batch, -1, -1).clone())
        for block in self.blocks:
            x, state = block(x, state)
        return self.head(self.norm(x).mean(dim=1))


for kind in MODELS:
    for count in ANCHOR_COUNTS:
        probe = AnchorViT(kind, count)
        print(f"{kind:7s} A={count:3d} {sum(p.numel() for p in probe.parameters()):>12,d} parameters")
        del probe

## Metrics

* **map_similarity**, **anchor_spread** — collapse metrics from experiment 03
* **object_mass** — average share of an anchor's pooling weight that lands on object
  patches, summed over anchors (1.0 = all anchors together spend one anchor's worth of
  attention on the object; uniform attention gives A × 6.25%)
* **object_anchor_share** — fraction of anchors whose region contains at least one
  object patch (positional models only)

In [ ]:
def off_diagonal_mean(similarity):
    size = similarity.shape[-1]
    mask = ~torch.eye(size, dtype=torch.bool, device=similarity.device)
    return similarity[:, mask].mean().item()


def router_stats(pooling, pooled, positions, on_object):
    pooling_unit = F.normalize(pooling, dim=-1)
    centred = pooled - pooled.mean(dim=1, keepdim=True)
    spread = centred.square().sum(dim=(1, 2)) / pooled.square().sum(dim=(1, 2)).clamp_min(1e-12)
    stats = {
        "map_similarity": off_diagonal_mean(pooling_unit @ pooling_unit.transpose(-2, -1)),
        "anchor_spread": spread.mean().item(),
        "object_mass": (pooling * on_object[:, None, :]).sum(dim=(1, 2)).mean().item(),
    }
    if positions is not None:
        coords = grid_positions(GRID).to(positions.device)
        owner = (positions[:, :, None, :] - coords[None, None]).square().sum(-1).argmin(dim=1)
        owns_object = F.one_hot(owner, positions.shape[1]) * on_object[:, :, None]
        stats["object_anchor_share"] = owns_object.amax(dim=1).float().mean().item()
    return stats


@torch.no_grad()
def anchor_metrics(model):
    model.eval()
    routers = model.routers()
    for router in routers:
        router.record = True
    totals = [defaultdict(float) for _ in routers]
    seen = 0
    for x, _, boxes in iterate_batches(test_x[:METRIC_IMAGES], test_y[:METRIC_IMAGES], train=False):
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            model(x)
        on_object = object_patch_mask(boxes)
        for total, router in zip(totals, routers):
            for name, value in router_stats(*router.last, on_object).items():
                total[name] += value * x.shape[0]
        seen += x.shape[0]
    for router in routers:
        router.record = False
        router.last = None
    return [{name: value / seen for name, value in total.items()} for total in totals]


@torch.no_grad()
def count_gflops(model, images):
    """Multiply-accumulates (as GFLOPs = 2 × MACs) per image of every Linear and Conv2d
    that runs, so patches that skip a layer are not counted. Attention score/value
    matmuls and the Voronoi assignment are not included; they are a few percent of the
    total at this size."""
    macs = 0

    def linear_hook(module, inputs, output):
        nonlocal macs
        macs += inputs[0].numel() // module.in_features * module.in_features * module.out_features

    def conv_hook(module, inputs, output):
        nonlocal macs
        kernel = module.kernel_size[0] * module.kernel_size[1]
        macs += output.numel() * module.in_channels * kernel // module.groups

    handles = []
    for module in model.modules():
        if isinstance(module, nn.Linear):
            handles.append(module.register_forward_hook(linear_hook))
        elif isinstance(module, nn.Conv2d):
            handles.append(module.register_forward_hook(conv_hook))
    model.eval()
    model(images)
    for handle in handles:
        handle.remove()
    return 2 * macs / images.shape[0] / 1e9

## Training

In [ ]:
@torch.no_grad()
def evaluate(model, criterion):
    model.eval()
    total_loss, correct, seen = 0.0, 0, 0
    for x, y, _ in iterate_batches(test_x, test_y, train=False):
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(x)
            loss = criterion(logits, y)
        total_loss += loss.item() * y.shape[0]
        correct += (logits.argmax(dim=1) == y).sum().item()
        seen += y.shape[0]
    return total_loss / seen, 100.0 * correct / seen


def run_experiment(kind, num_anchors):
    set_seed(SEED)
    model = AnchorViT(kind, num_anchors).to(DEVICE)
    params = sum(p.numel() for p in model.parameters())
    print(f"\n=== {kind} A={num_anchors} | {params:,} parameters ===")

    metrics_init = anchor_metrics(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=DEVICE.type == "cuda")
    generator = torch.Generator().manual_seed(SEED)

    history = []
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        start = time.perf_counter()
        total_loss, correct, seen = 0.0, 0, 0
        grad_norms, skipped = [], 0
        for x, y, _ in iterate_batches(train_x, train_y, train=True, generator=generator):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), GRAD_CLIP if GRAD_CLIP > 0 else float("inf")
            ).item()
            scaler.step(optimizer)
            scaler.update()
            if math.isfinite(grad_norm):
                grad_norms.append(grad_norm)
            else:
                skipped += 1
            total_loss += loss.item() * y.shape[0]
            correct += (logits.argmax(dim=1) == y).sum().item()
            seen += y.shape[0]
        scheduler.step()
        test_loss, test_acc = evaluate(model, criterion)
        minutes = (time.perf_counter() - start) / 60
        norms = np.array(grad_norms)
        history.append({"epoch": epoch, "train_loss": total_loss / seen, "train_acc": 100.0 * correct / seen,
                        "test_loss": test_loss, "test_acc": test_acc, "minutes": minutes,
                        "grad_norm_median": float(np.median(norms)), "grad_norm_max": float(norms.max()),
                        "steps_skipped": skipped})
        print(f"{kind} A={num_anchors} epoch {epoch:02d} | train acc {100.0 * correct / seen:5.2f}% | "
              f"test acc {test_acc:5.2f}% | grad norm median {np.median(norms):.2f} max {norms.max():.2f} | "
              f"{minutes:.2f} min")

    result = {"kind": kind, "num_anchors": num_anchors, "seed": SEED, "epochs": NUM_EPOCHS,
              "params": params, "history": history, "metrics_init": metrics_init,
              "metrics_trained": anchor_metrics(model), "gflops": count_gflops(model, sample_x[:16]),
              "keep_ratio": KEEP_RATIO, "skip_from": SKIP_FROM, "skip_score": SKIP_SCORE}
    if kind != "free":
        torch.save(model.state_dict(), os.path.join(WORK_DIR, f"{RUN_NAME}_{kind}_A{num_anchors}.pth"))
    del model, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

In [ ]:
results = {}
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as f:
        results = json.load(f)
    print("Loaded finished runs:", sorted(results))

for num_anchors in ANCHOR_COUNTS:
    for kind in MODELS:
        key = f"{kind}_A{num_anchors}"
        if key in results:
            print(f"Skipping {key}: already in {RESULTS_PATH}")
            continue
        results[key] = run_experiment(kind, num_anchors)
        with open(RESULTS_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)
        print("Saved:", RESULTS_PATH)

## Results

**Success criterion.** `moving` should match or beat `fixed` at the same anchor count,
and ideally `moving` with 4 anchors should approach `fixed` with 16, while keeping low
map similarity (no collapse) and putting more attention on the object than `fixed`. If
`moving` does not clearly beat `fixed` here, where the object is small and randomly
placed, the approach needs rethinking before any high-resolution work.

In [ ]:
def block_mean(blocks, name):
    values = [block[name] for block in blocks if name in block]
    return float(np.mean(values)) if values else float("nan")


lines = [
    "| Model | Anchors | Params | Final test acc | Best test acc | Map sim | Anchor spread | "
    "Object mass (last block) | Anchors touching object (last block) |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
]
for key, result in results.items():
    blocks = result["metrics_trained"]
    share = blocks[-1].get("object_anchor_share")
    lines.append(
        f"| {result['kind']} | {result['num_anchors']} | {result['params'] / 1e6:.2f}M | "
        f"{result['history'][-1]['test_acc']:.2f}% | {max(h['test_acc'] for h in result['history']):.2f}% | "
        f"{block_mean(blocks, 'map_similarity'):.3f} | {block_mean(blocks, 'anchor_spread'):.3f} | "
        f"{blocks[-1]['object_mass']:.3f} | {'' if share is None else f'{share:.2f}'} |"
    )
table = "\n".join(lines)
print(table)
with open(RESULTS_PATH.replace(".json", ".md"), "w", encoding="utf-8") as f:
    f.write(f"# Experiment 04 results ({DATASET}, seed {SEED}, {NUM_EPOCHS} epochs)\n\n{table}\n")

## Where do the anchors go?

For the trained `moving` and `fixed` models with the most anchors: test canvases with
the final-block regions overlaid, anchor positions per block (light → dark = block 1 → 6),
and the object box in white.

In [ ]:
import matplotlib.pyplot as plt


@torch.no_grad()
def anchor_trajectories(model, images):
    model.eval()
    routers = model.routers()
    for router in routers:
        router.record = True
    with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        model(images)
    positions = torch.stack([router.last[2] for router in routers], dim=1)
    for router in routers:
        router.record = False
        router.last = None
    return positions


def show_anchors(kind, num_anchors, count=4):
    path = os.path.join(WORK_DIR, f"{RUN_NAME}_{kind}_A{num_anchors}.pth")
    if not os.path.exists(path):
        print("No checkpoint for", kind, num_anchors)
        return
    model = AnchorViT(kind, num_anchors).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    images, boxes = sample_x[:count], sample_boxes[:count]
    positions = anchor_trajectories(model, images).cpu()
    coords = grid_positions(GRID)
    fig, axes = plt.subplots(1, count, figsize=(4 * count, 4))
    for i, ax in enumerate(axes):
        picture = (images[i] * STD[0] + MEAN[0]).clamp(0, 1).permute(1, 2, 0).cpu()
        ax.imshow(picture)
        final = positions[i, -1]
        owner = (final[:, None, :] - coords[None]).square().sum(-1).argmin(0).view(GRID, GRID)
        ax.imshow(owner, cmap="tab20", alpha=0.35, extent=(0, CANVAS, CANVAS, 0), interpolation="nearest")
        for block in range(positions.shape[1]):
            shade = plt.cm.Reds(0.3 + 0.7 * block / max(1, positions.shape[1] - 1))
            ax.scatter(positions[i, block, :, 1] * CANVAS, positions[i, block, :, 0] * CANVAS, s=12, color=shade)
        top, left, bottom, right = boxes[i].tolist()
        ax.add_patch(plt.Rectangle((left, top), right - left, bottom - top, fill=False, color="white", lw=2))
        ax.set_axis_off()
    fig.suptitle(f"{kind}, {num_anchors} anchors")
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, f"{RUN_NAME}_{kind}_A{num_anchors}_anchors.png"), dpi=110)
    plt.show()
    del model


for kind in ("focus", "moving", "fixed"):
    if kind in MODELS:
        show_anchors(kind, max(ANCHOR_COUNTS))